# Local ST Spectrum Diagnostics: July 2024, 2 Random Days

Lightweight local notebook based on `real_july_st_corridor_spectral_profile_v2_061426.py`.

- Uses 2024 July only.
- Randomly selects 2 July days with a fixed seed.
- Fits the full 4x4 corridor ST Vecchia model with lag pattern 6/4/3.
- Nugget is fixed at 0.
- Compares baseline Matérn `smooth=0.3` against the 2024 GC candidate `alpha=0.8, beta=1`.
- Keeps only norm-frequency summaries, not latitude/longitude/diagonal directional plots.

Important: this diagnostic uses spatial 2D FFT for each of the 8 time slots and then an 8x8 finite-sample cross-periodogram whitening step. It is not a 3D FFT diagnostic.

In [1]:
from __future__ import annotations

import importlib.util
import json
import math
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

%matplotlib inline

REPO = Path("/Users/joonwonlee/Documents/GEMS_TCO-1")
SRC = REPO / "src"
SCRIPT_PATH = REPO / "Exercises/st_model/day/amarel_simulation/space_time/spectral_analysis/real_july_st_corridor_spectral_profile_v2_061426.py"
NOTEBOOK_DIR = REPO / "Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics"
OUT_DIR = NOTEBOOK_DIR / "outputs" / "real_july2024_st_spectral_norm_2days_matern_gc_nugget0_061626"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [str(SRC), str(SCRIPT_PATH.parent), str(SCRIPT_PATH.parent.parent), str(SCRIPT_PATH.parent.parent / "real_data"), str(SCRIPT_PATH.parent.parent / "vecchia_diagnosis")]:
    if p not in sys.path:
        sys.path.insert(0, p)

spec = importlib.util.spec_from_file_location("st_spectral_v2", SCRIPT_PATH)
st = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = st
spec.loader.exec_module(st)

print("Loaded:", SCRIPT_PATH)
print("Output:", OUT_DIR)
print("2D FFT check: expected_periodogram_raw_st uses torch.fft.fft2(..., dim=(0, 1)); time is handled as an 8x8 cross-periodogram.")

Loaded: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/amarel_simulation/space_time/spectral_analysis/real_july_st_corridor_spectral_profile_v2_061426.py
Output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_spectral_norm_2days_matern_gc_nugget0_061626
2D FFT check: expected_periodogram_raw_st uses torch.fft.fft2(..., dim=(0, 1)); time is handled as an 8x8 cross-periodogram.


In [2]:
# ---------------------------
# Lightweight local settings
# ---------------------------
YEAR = 2024
MONTH = 7
RANDOM_SEED = 20260616
N_RANDOM_DAYS = 2
ALL_DAY_IDXS = np.arange(30)  # July day_idx 0..29
DAY_IDXS = sorted(np.random.default_rng(RANDOM_SEED).choice(ALL_DAY_IDXS, size=N_RANDOM_DAYS, replace=False).astype(int).tolist())

# 2024 final candidate. Add "gc_a075_b1" here if you want a quick sensitivity check.
MODEL_VARIANTS = ["matern_s03", "gc_a08_b1"]

args = st.build_arg_parser().parse_args([])
args.real_years = [str(YEAR)]
args.month = MONTH
# The source helper treats a two-token string like "a,b" as range(a,b).
# load_real_assets was imported from the diagnosis module, so patch its own globals.
_exact_two_day_parser = lambda _text: list(DAY_IDXS)
st.parse_day_idxs = _exact_two_day_parser
st.load_real_assets.__globals__["parse_day_idxs"] = _exact_two_day_parser
args.days = "0,30"  # ignored by patched parser; kept parseable for safety
args.model_variants = MODEL_VARIANTS
args.out_dir = OUT_DIR
args.monthly_out_dir = OUT_DIR / "monthly_plots_top"
args.suppress_fit_prints = True
args.n_bins = 80
args.skip_existing = False
args.device = None
args.require_cuda = False
args.cuda_fallback = "cpu"

# Keep the same fitting defaults as the ST spectral script, but the run is only 2 days x 2 models.
print("Selected day_idx:", DAY_IDXS)
print("Selected dates:", [f"{YEAR}-{MONTH:02d}-{d + 1:02d}" for d in DAY_IDXS])
print("Models:", MODEL_VARIANTS)


Selected day_idx: [0, 4]
Selected dates: ['2024-07-01', '2024-07-05']
Models: ['matern_s03', 'gc_a08_b1']


In [3]:
# Load just the selected two real-data daily assets.
# If local data are not in the default repo/data location, set args.data_root before this cell.
device = st.resolve_device(args)
print("Device:", device)

assets = st.load_real_assets(args)
print("Loaded assets:", [(a["day"], len(a["source_map"]), a["grid_coords_np"].shape) for a in assets])

Device: cpu

Loading real July data for 2024-07
--- Global Monthly Mean for 2024-7: 257.9726 ---
n hourly slots: 248 grid: (18126, 2) monthly_mean: 257.9726104252314
Loaded assets: [('2024-07-01', 8, (18126, 2)), ('2024-07-05', 8, (18126, 2))]


In [4]:
fit_rows = []
profile_rows = []
fit_id = 0

for asset in assets:
    for model_variant in MODEL_VARIANTS:
        fit_id += 1
        model_spec = st.variant_spec(model_variant)
        print("\n" + "-" * 90)
        print(f"fit_id={fit_id} day={asset['day']} model={model_variant} ({model_spec['label']})")
        print("-" * 90)
        t0 = time.time()
        row, beta, lat_mean = st.fit_full_asset(
            asset=asset,
            spec=model_spec,
            init_physical=st.DEFAULT_REAL_INIT_PHYSICAL,
            reference_advec_lon_abs=float(args.real_reference_advec_lon_abs),
            fit_id=fit_id,
            device=device,
            args=args,
        )
        est = {k: float(row[f"est_{k}"]) for k in st.P_LABELS}
        est["loss"] = float(row["loss"])
        rows, stats = st.compute_spectral_profile(
            asset=asset,
            est=est,
            beta=beta,
            lat_mean=lat_mean,
            spec=model_spec,
            fit_id=fit_id,
            device=device,
            args=args,
        )
        row.update(stats)
        fit_rows.append(row)
        profile_rows.extend([r for r in rows if r.get("direction") == "norm"])
        print(
            f"done in {time.time() - t0:.1f}s | "
            f"loss={row['loss']:.5f} | "
            f"sigmasq={row['est_sigmasq']:.4g} "
            f"range_time={row['est_range_time']:.4g}"
        )

fit_df = pd.DataFrame(fit_rows)
profile_df = pd.DataFrame(profile_rows)
fit_df.to_csv(OUT_DIR / "local_st_2day_fit_rows.csv", index=False)
profile_df.to_csv(OUT_DIR / "local_st_2day_norm_profile_rows.csv", index=False)

fit_df[["fit_id", "day", "model_variant", "model_label", "loss", "est_sigmasq", "est_range_lat", "est_range_lon", "est_range_time", "est_advec_lat", "est_advec_lon", "est_nugget"]]



------------------------------------------------------------------------------------------
fit_id=1 day=2024-07-01 model=matern_s03 (Matern s=0.3 nugget0)
------------------------------------------------------------------------------------------
Pre-computing StrategyClusterVecchia (smooth=0.3, strategy=offset_corridor_tapered, block=(4, 4), origin=0/0, lag_blocks=6/4/3, basis=corridor, force_center=0, offsets=0.1260/0.2520, corridors=(0.063, 0.189)/(0.0, 0.252), anchor_mode=width)... Done. clusters=1160, max_points/block=16, target_blocks=8896, target_points=131428, batches=[A:m96:b4x1, A:m96:b8x52, A:m96:b10x1, A:m96:b12x14, A:m96:b16x1092, AB:m160:b1x16, AB:m160:b2x12, AB:m160:b3x8, AB:m160:b4x10, AB:m160:b5x5, ... (37 batches)]
done in 602.7s | loss=1.19175 | sigmasq=14.72 range_time=2.328

------------------------------------------------------------------------------------------
fit_id=2 day=2024-07-01 model=gc_a08_b1 (GC a=0.8 b=1 nugget0)
---------------------------------------

,fit_id,day,model_variant,model_label,loss,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget
0,1,2024-07-01,matern_s03,Matern s=0.3 nugget0,1.191750,14.723408,0.417327,0.483928,2.327765,-0.017328,-0.196924,0.0
1,2,2024-07-01,gc_a08_b1,GC a=0.8 b=1 nugget0,1.189021,18.550840,0.450202,0.527239,2.654157,-0.016499,-0.197460,0.0
2,3,2024-07-05,matern_s03,Matern s=0.3 nugget0,1.352562,15.997284,0.265737,0.327968,2.027016,0.054500,-0.216797,0.0
3,4,2024-07-05,gc_a08_b1,GC a=0.8 b=1 nugget0,1.349534,18.521111,0.264631,0.327481,2.051050,0.054364,-0.216553,0.0


In [5]:
# Monthly-style summary, but only over the two sampled days and only norm frequency.
monthly = st.make_profile_monthly_summary(profile_df)
monthly = monthly[monthly["direction"] == "norm"].copy()
monthly.to_csv(OUT_DIR / "local_st_2day_norm_monthly_summary.csv", index=False)

band_table = st.make_ratio_band_table(monthly)
band_table.to_csv(OUT_DIR / "local_st_2day_norm_band_table.csv", index=False)

# Loss table with 5 decimals for direct model comparison.
loss_summary = (
    fit_df.groupby(["model_variant", "model_label"], as_index=False)
    .agg(loss_mean=("loss", "mean"), loss_median=("loss", "median"), n_days=("loss", "size"))
)
loss_summary["loss_mean"] = loss_summary["loss_mean"].map(lambda x: f"{x:.5f}")
loss_summary["loss_median"] = loss_summary["loss_median"].map(lambda x: f"{x:.5f}")
loss_summary.to_csv(OUT_DIR / "local_st_2day_loss_summary.csv", index=False)
loss_summary

,model_variant,model_label,loss_mean,loss_median,n_days
0,gc_a08_b1,GC a=0.8 b=1 nugget0,1.26928,1.26928,2
1,matern_s03,Matern s=0.3 nugget0,1.27216,1.27216,2


In [6]:
def label_with_loss(df: pd.DataFrame) -> str:
    label = str(df["model_label"].dropna().iloc[0])
    loss = pd.to_numeric(df["loss_median"], errors="coerce").median()
    return f"{label} loss={loss:.5f}" if np.isfinite(loss) else label

METRICS = [
    ("ratio_I_over_EI_profile_mean", "I / E[I] (profile sigma)", "data finite-sample spatial spectrum / fitted E[I]"),
    ("ratio_EI_over_continuous_mean", "E[I] / continuous", "finite-sample fitted E[I] / continuous-like spectrum"),
    ("whitened_ratio_mean", "8x8 whitened I / E[I]", "8x8 Cholesky-whitened quadratic power / profiled target"),
]
BANDS = {
    "ratio_I_over_EI_profile_mean": ("ratio_I_over_EI_profile_p10", "ratio_I_over_EI_profile_p90"),
    "ratio_EI_over_continuous_mean": ("ratio_EI_over_continuous_p10", "ratio_EI_over_continuous_p90"),
    "whitened_ratio_mean": ("whitened_ratio_p10", "whitened_ratio_p90"),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), sharex=True)
for ax, (metric, title, ylabel) in zip(axes, METRICS):
    for model_variant, sub in monthly.groupby("model_variant", dropna=False):
        sub = sub.sort_values("k_mid")
        line = ax.plot(sub["k_mid"], sub[metric], linewidth=2.0, label=label_with_loss(sub))[0]
        lo_col, hi_col = BANDS[metric]
        if lo_col in sub.columns and hi_col in sub.columns:
            x = pd.to_numeric(sub["k_mid"], errors="coerce").to_numpy(float)
            lo = pd.to_numeric(sub[lo_col], errors="coerce").to_numpy(float)
            hi = pd.to_numeric(sub[hi_col], errors="coerce").to_numpy(float)
            ok = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi) & (lo > 0) & (hi > 0)
            ax.fill_between(x[ok], lo[ok], hi[ok], color=line.get_color(), alpha=0.14, linewidth=0)
    ax.axhline(1.0, color="0.25", linestyle="--", linewidth=1.0)
    ax.set_title(title)
    ax.set_xlabel("norm frequency")
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.set_ylim(0.2, 5.0)
    ax.grid(alpha=0.25, which="both")
axes[0].legend(fontsize=8)
fig.suptitle(f"July {YEAR}: ST corridor norm-frequency spectral ratios, 2 sampled days {DAY_IDXS}")
fig.tight_layout()
plot_path = OUT_DIR / "local_st_2day_norm_ratio_panels.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)

Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_spectral_norm_2days_matern_gc_nugget0_061626/local_st_2day_norm_ratio_panels.png


/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_65059/3390740805.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Representative frequency bands for quick numeric checks.
# lowest_frequency_bin: bin 0
# low_frequency_bins_1_5: bins 1-5
# mid/high: automatic relative bands from the source diagnostic helper.
cols = [
    "model_label", "frequency_band", "k_mid_mean", "n_bins",
    "ratio_I_over_EI_profile_mean", "ratio_EI_over_continuous_mean", "whitened_ratio_mean",
]
if not band_table.empty:
    display(band_table[band_table["direction"] == "norm"][cols].sort_values(["model_label", "frequency_band"]))
else:
    print("No band table rows produced.")

,model_label,frequency_band,k_mid_mean,n_bins,ratio_I_over_EI_profile_mean,ratio_EI_over_continuous_mean,whitened_ratio_mean
0,GC a=0.8 b=1 nugget0,high_frequency_band,12.449007,16,0.863516,1.052167,0.865260
1,GC a=0.8 b=1 nugget0,low_frequency_bins_1_5,0.605160,5,1.090023,1.080385,0.913375
2,GC a=0.8 b=1 nugget0,lowest_frequency_bin,0.086451,1,0.998643,0.743734,1.177823
3,GC a=0.8 b=1 nugget0,mid_frequency_band,6.224503,16,1.055072,1.091085,1.056272
4,Matern s=0.3 nugget0,high_frequency_band,12.449007,16,0.808890,1.037089,0.808358
5,Matern s=0.3 nugget0,low_frequency_bins_1_5,0.605160,5,1.101578,0.979110,0.885661
6,Matern s=0.3 nugget0,lowest_frequency_bin,0.086451,1,2.311320,0.819170,1.439711
7,Matern s=0.3 nugget0,mid_frequency_band,6.224503,16,1.083327,1.067017,1.079116


## Notes

`I / E[I]` here uses the profile-sigma version, matching the recent ST ratio plots. The raw `I / E[I]` without profile scaling is still saved in `local_st_2day_norm_profile_rows.csv` as `ratio_I_over_EI_mean` if you want to inspect it directly.

The FFT step is spatial 2D FFT only. The time dimension remains an 8-variate Fourier vector, and the fitted 8x8 finite-sample cross-periodogram is used for whitening.